In [1]:
import pandas as pd

In [2]:
# Load the data files
prices_df = pd.read_csv('prices_daily_tidy_clean.csv')
ump_df = pd.read_csv('ump_dki_jakarta_daily_clean.csv')

# Display the first few rows
print("Prices data:")
print(prices_df.head())
print("\nUMP data:")
print(ump_df.head())

Prices data:
      tanggal         region komoditas    harga  harga_is_outlier  \
0  2024-01-01  Jakarta Barat  ayam_ras  38750.0                 0   
1  2024-01-02  Jakarta Barat  ayam_ras  39429.0                 0   
2  2024-01-03  Jakarta Barat  ayam_ras  39571.0                 0   
3  2024-01-04  Jakarta Barat  ayam_ras  38667.0                 0   
4  2024-01-05  Jakarta Barat  ayam_ras  40000.0                 0   

   harga_is_imputed  
0                 0  
1                 0  
2                 0  
3                 0  
4                 0  

UMP data:
      tanggal         region    ump_daily      ump
0  01/01/2024  Jakarta Barat  163463.9032  5067381
1  02/01/2024  Jakarta Barat  163463.9032  5067381
2  03/01/2024  Jakarta Barat  163463.9032  5067381
3  04/01/2024  Jakarta Barat  163463.9032  5067381
4  05/01/2024  Jakarta Barat  163463.9032  5067381


In [3]:
prices_df.head()

,tanggal,region,komoditas,harga,harga_is_outlier,harga_is_imputed
0,2024-01-01,Jakarta Barat,ayam_ras,38750.0,0,0
1,2024-01-02,Jakarta Barat,ayam_ras,39429.0,0,0
2,2024-01-03,Jakarta Barat,ayam_ras,39571.0,0,0
3,2024-01-04,Jakarta Barat,ayam_ras,38667.0,0,0
4,2024-01-05,Jakarta Barat,ayam_ras,40000.0,0,0


In [4]:
ump_df.head()

,tanggal,region,ump_daily,ump
0,01/01/2024,Jakarta Barat,163463.9032,5067381
1,02/01/2024,Jakarta Barat,163463.9032,5067381
2,03/01/2024,Jakarta Barat,163463.9032,5067381
3,04/01/2024,Jakarta Barat,163463.9032,5067381
4,05/01/2024,Jakarta Barat,163463.9032,5067381


In [5]:
# Parse date columns properly
prices_df["tanggal"] = pd.to_datetime(prices_df["tanggal"], format='%Y-%m-%d')
ump_df["tanggal"] = pd.to_datetime(ump_df["tanggal"], dayfirst=True)

print("Prices data types:")
print(prices_df.dtypes)
print("\nUMP data types:")
print(ump_df.dtypes)
print("\nPrices dates sample:")
print(prices_df["tanggal"].head())
print("\nUMP dates sample:")
print(ump_df["tanggal"].head())

Prices data types:
tanggal             datetime64[ns]
region                      object
komoditas                   object
harga                      float64
harga_is_outlier             int64
harga_is_imputed             int64
dtype: object

UMP data types:
tanggal      datetime64[ns]
region               object
ump_daily           float64
ump                   int64
dtype: object

Prices dates sample:
0   2024-01-01
1   2024-01-02
2   2024-01-03
3   2024-01-04
4   2024-01-05
Name: tanggal, dtype: datetime64[ns]

UMP dates sample:
0   2024-01-01
1   2024-01-02
2   2024-01-03
3   2024-01-04
4   2024-01-05
Name: tanggal, dtype: datetime64[ns]


In [6]:
# Sort and ensure 1 row per day per series

# For prices_df: group by (region, komoditas, tanggal) and take mean
print("Prices df - checking for duplicates per (region, komoditas, tanggal)...")
prices_duplicates = prices_df.groupby(['region', 'komoditas', 'tanggal']).size()
duplicate_count_prices = (prices_duplicates > 1).sum()
print(f"Found {duplicate_count_prices} duplicates")

# Aggregate by taking mean for numeric columns
prices_df = prices_df.groupby(['region', 'komoditas', 'tanggal'], as_index=False).agg({
    'harga': 'mean',
    'harga_is_outlier': 'first',
    'harga_is_imputed': 'first'
}).sort_values(['region', 'komoditas', 'tanggal']).reset_index(drop=True)

# For ump_df: group by (region, tanggal) and take mean
print("\nUMP df - checking for duplicates per (region, tanggal)...")
ump_duplicates = ump_df.groupby(['region', 'tanggal']).size()
duplicate_count_ump = (ump_duplicates > 1).sum()
print(f"Found {duplicate_count_ump} duplicates")

# Aggregate by taking mean for numeric columns
ump_df = ump_df.groupby(['region', 'tanggal'], as_index=False).agg({
    'ump_daily': 'mean',
    'ump': 'mean'
}).sort_values(['region', 'tanggal']).reset_index(drop=True)

print("\nPrices shape after deduplication:", prices_df.shape)
print("UMP shape after deduplication:", ump_df.shape)
print("\nPrices sample:")
print(prices_df.head(10))
print("\nUMP sample:")
print(ump_df.head(10))

Prices df - checking for duplicates per (region, komoditas, tanggal)...
Found 0 duplicates

UMP df - checking for duplicates per (region, tanggal)...
Found 0 duplicates

Prices shape after deduplication: (36550, 6)
UMP shape after deduplication: (3655, 4)

Prices sample:
          region komoditas    tanggal    harga  harga_is_outlier  \
0  Jakarta Barat  ayam_ras 2024-01-01  38750.0                 0   
1  Jakarta Barat  ayam_ras 2024-01-02  39429.0                 0   
2  Jakarta Barat  ayam_ras 2024-01-03  39571.0                 0   
3  Jakarta Barat  ayam_ras 2024-01-04  38667.0                 0   
4  Jakarta Barat  ayam_ras 2024-01-05  40000.0                 0   
5  Jakarta Barat  ayam_ras 2024-01-06  39429.0                 0   
6  Jakarta Barat  ayam_ras 2024-01-07  38000.0                 0   
7  Jakarta Barat  ayam_ras 2024-01-08  39429.0                 0   
8  Jakarta Barat  ayam_ras 2024-01-09  40000.0                 0   
9  Jakarta Barat  ayam_ras 2024-01-10  38667.0  

In [7]:
# Reindex to full daily calendar and fill missing values

# For prices_df: reindex by (region, komoditas)
print("Reindexing prices_df to full daily calendar...")
prices_list = []
for (region, komoditas), group in prices_df.groupby(['region', 'komoditas']):
    # Set index to tanggal
    group = group.set_index('tanggal')
    # Create full date range
    date_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    # Reindex
    group = group.reindex(date_range)
    # Fill missing: ffill then bfill
    group['harga'] = group['harga'].ffill().bfill()
    group['harga_is_outlier'] = group['harga_is_outlier'].ffill().bfill()
    group['harga_is_imputed'] = group['harga_is_imputed'].ffill().bfill()
    # Reset index and restore region, komoditas columns
    group['region'] = region
    group['komoditas'] = komoditas
    group = group.reset_index().rename(columns={'index': 'tanggal'})
    prices_list.append(group)

prices_df = pd.concat(prices_list, ignore_index=True)
prices_df = prices_df[['region', 'komoditas', 'tanggal', 'harga', 'harga_is_outlier', 'harga_is_imputed']]
prices_df = prices_df.sort_values(['region', 'komoditas', 'tanggal']).reset_index(drop=True)

# For ump_df: reindex by (region)
print("Reindexing ump_df to full daily calendar...")
ump_list = []
for region, group in ump_df.groupby('region'):
    # Set index to tanggal
    group = group.set_index('tanggal')
    # Create full date range
    date_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    # Reindex
    group = group.reindex(date_range)
    # Fill missing: ffill only (stepwise data)
    group['ump_daily'] = group['ump_daily'].ffill()
    group['ump'] = group['ump'].ffill()
    # Reset index and restore region column
    group['region'] = region
    group = group.reset_index().rename(columns={'index': 'tanggal'})
    ump_list.append(group)

ump_df = pd.concat(ump_list, ignore_index=True)
ump_df = ump_df[['region', 'tanggal', 'ump_daily', 'ump']]
ump_df = ump_df.sort_values(['region', 'tanggal']).reset_index(drop=True)

print(f"\nPrices shape after reindex: {prices_df.shape}")
print(f"UMP shape after reindex: {ump_df.shape}")
print("\nPrices sample:")
print(prices_df[prices_df['komoditas'] == 'ayam_ras'].head(15))
print("\nUMP sample:")
print(ump_df.head(15))

Reindexing prices_df to full daily calendar...
Reindexing ump_df to full daily calendar...

Prices shape after reindex: (36550, 6)
UMP shape after reindex: (3655, 4)

Prices sample:
           region komoditas    tanggal    harga  harga_is_outlier  \
0   Jakarta Barat  ayam_ras 2024-01-01  38750.0                 0   
1   Jakarta Barat  ayam_ras 2024-01-02  39429.0                 0   
2   Jakarta Barat  ayam_ras 2024-01-03  39571.0                 0   
3   Jakarta Barat  ayam_ras 2024-01-04  38667.0                 0   
4   Jakarta Barat  ayam_ras 2024-01-05  40000.0                 0   
5   Jakarta Barat  ayam_ras 2024-01-06  39429.0                 0   
6   Jakarta Barat  ayam_ras 2024-01-07  38000.0                 0   
7   Jakarta Barat  ayam_ras 2024-01-08  39429.0                 0   
8   Jakarta Barat  ayam_ras 2024-01-09  40000.0                 0   
9   Jakarta Barat  ayam_ras 2024-01-10  38667.0                 0   
10  Jakarta Barat  ayam_ras 2024-01-11  39571.0            

In [8]:
# Merge prices with UMP (exogenous feature)

# Select only ump column (baseline approach)
ump_merged = ump_df[['region', 'tanggal', 'ump']].copy()

# Merge on tanggal and region
df_merged = prices_df.merge(ump_merged, on=['region', 'tanggal'], how='left')

print(f"Prices shape: {prices_df.shape}")
print(f"UMP shape (subset): {ump_merged.shape}")
print(f"Merged shape: {df_merged.shape}")
print(f"\nMerged columns: {df_merged.columns.tolist()}")
print(f"\nCheck for missing values after merge:")
print(df_merged.isnull().sum())
print(f"\nMerged data sample:")
print(df_merged[df_merged['komoditas'] == 'ayam_ras'].head(10))

Prices shape: (36550, 6)
UMP shape (subset): (3655, 3)
Merged shape: (36550, 7)

Merged columns: ['region', 'komoditas', 'tanggal', 'harga', 'harga_is_outlier', 'harga_is_imputed', 'ump']

Check for missing values after merge:
region              0
komoditas           0
tanggal             0
harga               0
harga_is_outlier    0
harga_is_imputed    0
ump                 0
dtype: int64

Merged data sample:
          region komoditas    tanggal    harga  harga_is_outlier  \
0  Jakarta Barat  ayam_ras 2024-01-01  38750.0                 0   
1  Jakarta Barat  ayam_ras 2024-01-02  39429.0                 0   
2  Jakarta Barat  ayam_ras 2024-01-03  39571.0                 0   
3  Jakarta Barat  ayam_ras 2024-01-04  38667.0                 0   
4  Jakarta Barat  ayam_ras 2024-01-05  40000.0                 0   
5  Jakarta Barat  ayam_ras 2024-01-06  39429.0                 0   
6  Jakarta Barat  ayam_ras 2024-01-07  38000.0                 0   
7  Jakarta Barat  ayam_ras 2024-01-08  39

In [9]:
# Split train/val/test based on time (80/10/10)

# Split per (region, komoditas) series to maintain temporal order
train_list = []
val_list = []
test_list = []

for (region, komoditas), group in df_merged.groupby(['region', 'komoditas']):
    n = len(group)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    
    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])

train_data = pd.concat(train_list, ignore_index=True)
val_data = pd.concat(val_list, ignore_index=True)
test_data = pd.concat(test_list, ignore_index=True)

print("Train/Val/Test Split (Time-based 80/10/10):\n")
print(f"Train shape: {train_data.shape}")
print(f"Val shape: {val_data.shape}")
print(f"Test shape: {test_data.shape}")
print(f"Total: {train_data.shape[0] + val_data.shape[0] + test_data.shape[0]}")
print(f"\nPercentages:")
print(f"Train: {100*train_data.shape[0]/(train_data.shape[0]+val_data.shape[0]+test_data.shape[0]):.2f}%")
print(f"Val: {100*val_data.shape[0]/(train_data.shape[0]+val_data.shape[0]+test_data.shape[0]):.2f}%")
print(f"Test: {100*test_data.shape[0]/(train_data.shape[0]+val_data.shape[0]+test_data.shape[0]):.2f}%")

print(f"\nTrain date range: {train_data['tanggal'].min()} to {train_data['tanggal'].max()}")
print(f"Val date range: {val_data['tanggal'].min()} to {val_data['tanggal'].max()}")
print(f"Test date range: {test_data['tanggal'].min()} to {test_data['tanggal'].max()}")

print("\nTrain sample:")
print(train_data.tail(5))
print("\nVal sample:")
print(val_data.head(5))
print("\nTest sample:")
print(test_data.tail(5))

Train/Val/Test Split (Time-based 80/10/10):

Train shape: (29200, 7)
Val shape: (3650, 7)
Test shape: (3700, 7)
Total: 36550

Percentages:
Train: 79.89%
Val: 9.99%
Test: 10.12%

Train date range: 2024-01-01 00:00:00 to 2025-08-06 00:00:00
Val date range: 2025-08-07 00:00:00 to 2025-10-18 00:00:00
Test date range: 2025-10-19 00:00:00 to 2025-12-31 00:00:00

Train sample:
              region       komoditas    tanggal    harga  harga_is_outlier  \
29195  Jakarta Utara  telur_ayam_ras 2025-08-02  29667.0                 0   
29196  Jakarta Utara  telur_ayam_ras 2025-08-03  29600.0                 0   
29197  Jakarta Utara  telur_ayam_ras 2025-08-04  29250.0                 0   
29198  Jakarta Utara  telur_ayam_ras 2025-08-05  29000.0                 0   
29199  Jakarta Utara  telur_ayam_ras 2025-08-06  29250.0                 0   

       harga_is_imputed        ump  
29195                 0  5396761.0  
29196                 0  5396761.0  
29197                 0  5396761.0  
29198     

In [10]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Scaling without data leakage: fit scaler only on train data
# Features to scale: [harga, ump]
feature_cols = ['harga', 'ump']

# Initialize scaler
scaler = StandardScaler()

# Fit scaler ONLY on train data
scaler.fit(train_data[feature_cols])

# Transform train/val/test using train scaler
train_scaled = train_data.copy()
val_scaled = val_data.copy()
test_scaled = test_data.copy()

train_scaled[feature_cols] = scaler.transform(train_data[feature_cols])
val_scaled[feature_cols] = scaler.transform(val_data[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_data[feature_cols])

print("Scaling without data leakage:\n")
print(f"Scaler fitted on train data only")
print(f"Features scaled: {feature_cols}")
print(f"\nScaler mean: {scaler.mean_}")
print(f"Scaler scale: {scaler.scale_}")

print(f"\n--- Train data (scaled) ---")
print(train_scaled[feature_cols + ['region', 'komoditas']].head(10))
print(f"\n--- Val data (scaled) ---")
print(val_scaled[feature_cols + ['region', 'komoditas']].head(5))
print(f"\n--- Test data (scaled) ---")
print(test_scaled[feature_cols + ['region', 'komoditas']].head(5))

print(f"\nValue ranges after scaling:")
print(f"Train harga - min: {train_scaled['harga'].min():.4f}, max: {train_scaled['harga'].max():.4f}")
print(f"Train ump - min: {train_scaled['ump'].min():.4f}, max: {train_scaled['ump'].max():.4f}")
print(f"Val harga - min: {val_scaled['harga'].min():.4f}, max: {val_scaled['harga'].max():.4f}")
print(f"Val ump - min: {val_scaled['ump'].min():.4f}, max: {val_scaled['ump'].max():.4f}")
print(f"Test harga - min: {test_scaled['harga'].min():.4f}, max: {test_scaled['harga'].max():.4f}")
print(f"Test ump - min: {test_scaled['ump'].min():.4f}, max: {test_scaled['ump'].max():.4f}")

Scaling without data leakage:

Scaler fitted on train data only
Features scaled: ['harga', 'ump']

Scaler mean: [  46839.45712329 5190334.49315068]
Scaler scale: [ 35915.89266494 159313.71596952]

--- Train data (scaled) ---
      harga      ump         region komoditas
0 -0.225233 -0.77177  Jakarta Barat  ayam_ras
1 -0.206328 -0.77177  Jakarta Barat  ayam_ras
2 -0.202374 -0.77177  Jakarta Barat  ayam_ras
3 -0.227544 -0.77177  Jakarta Barat  ayam_ras
4 -0.190430 -0.77177  Jakarta Barat  ayam_ras
5 -0.206328 -0.77177  Jakarta Barat  ayam_ras
6 -0.246115 -0.77177  Jakarta Barat  ayam_ras
7 -0.206328 -0.77177  Jakarta Barat  ayam_ras
8 -0.190430 -0.77177  Jakarta Barat  ayam_ras
9 -0.227544 -0.77177  Jakarta Barat  ayam_ras

--- Val data (scaled) ---
      harga       ump         region komoditas
0 -0.265995  1.295723  Jakarta Barat  ayam_ras
1 -0.265995  1.295723  Jakarta Barat  ayam_ras
2 -0.236844  1.295723  Jakarta Barat  ayam_ras
3 -0.238152  1.295723  Jakarta Barat  ayam_ras
4 -0.23

In [15]:
import numpy as np

def make_sequence(data, lookback=30, horizon=1, target_idx=0):
    """Convert a time-series array into (X, y) sequences."""
    arr = np.asarray(data)
    X, y = [], []
    for t in range(lookback, len(arr) - horizon + 1):
        X.append(arr[t-lookback:t])
        if arr.ndim > 1:
            y.append(arr[t + horizon - 1, target_idx])
        else:
            y.append(arr[t + horizon - 1])
    if len(X) > 0:
        X = np.stack(X)
    else:
        n_features = 1 if arr.ndim == 1 else arr.shape[1]
        X = np.empty((0, lookback, n_features))
    y = np.array(y).reshape(-1, 1)
    return X, y

# Note: `data` can be shape (n,) for univariate or (n, n_features) for multivariate.
# `target_idx` selects the column in `data` used as the prediction target (default 0).


In [16]:
import numpy as np

FEAT_COLS = ["harga", "ump"]   # pastikan ini sama dengan scaler-fit
TARGET_IDX = 0                # 'harga' di kolom pertama
LOOKBACK = 30
HORIZON = 1

def make_sequence(arr, lookback=30, horizon=1, target_idx=0):
    arr = np.asarray(arr, dtype=float)
    X, y = [], []
    for t in range(lookback, len(arr) - horizon + 1):
        X.append(arr[t-lookback:t, :])
        y.append(arr[t + horizon - 1, target_idx])
    if len(X) == 0:
        return np.empty((0, lookback, arr.shape[1])), np.empty((0, 1))
    return np.stack(X), np.asarray(y, dtype=float).reshape(-1, 1)

def build_Xy_from_grouped(df_scaled, feat_cols, lookback=30, horizon=1, target_idx=0):
    X_list, y_list = [], []
    for (region, komoditas), g in df_scaled.groupby(["region", "komoditas"]):
        g = g.sort_values("tanggal").reset_index(drop=True)
        arr = g[feat_cols].astype(float).values
        Xg, yg = make_sequence(arr, lookback=lookback, horizon=horizon, target_idx=target_idx)
        if Xg.shape[0] == 0:
            continue
        X_list.append(Xg)
        y_list.append(yg)
    if len(X_list) == 0:
        return np.empty((0, lookback, len(feat_cols))), np.empty((0, 1))
    return np.vstack(X_list), np.vstack(y_list)

In [17]:
# Example usage: create sequences from the first available (region, komoditas) in `train_scaled`
pair = train_scaled[['region','komoditas']].drop_duplicates().iloc[0]
region, komoditas = pair['region'], pair['komoditas']
print(f"Using series: {region} - {komoditas}")
series = train_scaled[(train_scaled['region']==region) & (train_scaled['komoditas']==komoditas)]
# use 'harga' as target (col 0) and include 'ump' as exogenous feature -> shape (n, 2)
arr = series[['harga','ump']].values
X, y = make_sequence(arr, lookback=30, horizon=1, target_idx=0)
print('X.shape, y.shape ->', X.shape, y.shape)


X_all, y_all = make_sequence(train_scaled[['harga','ump']].values, lookback=30, horizon=1, target_idx=0)
print('All-train X_all.shape, y_all.shape ->', X_all.shape, y_all.shape)


Using series: Jakarta Barat - ayam_ras
X.shape, y.shape -> (554, 30, 2) (554, 1)
All-train X_all.shape, y_all.shape -> (29170, 30, 2) (29170, 1)


In [19]:
X_train, y_train = build_Xy_from_grouped(train_scaled, FEAT_COLS, LOOKBACK, HORIZON, TARGET_IDX)
X_val,   y_val   = build_Xy_from_grouped(val_scaled,   FEAT_COLS, LOOKBACK, HORIZON, TARGET_IDX)
X_test,  y_test  = build_Xy_from_grouped(test_scaled,  FEAT_COLS, LOOKBACK, HORIZON, TARGET_IDX)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(27700, 30, 2) (27700, 1)
(2150, 30, 2) (2150, 1)
(2200, 30, 2) (2200, 1)


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Baseline LSTM builder
def build_baseline_lstm(input_shape, lstm_units=32, dropout_rate=0.1):
    model = Sequential()
    model.add(LSTM(lstm_units, input_shape=input_shape))
    if dropout_rate and dropout_rate > 0:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    model.compile(optimizer=Adam(), loss='mse', metrics=['mae'])
    return model

# Try to build model automatically if `X` exists
try:
    model = build_baseline_lstm(input_shape=(X.shape[1], X.shape[2]))
    model.summary()
except Exception as e:
    print('Could not auto-build model (X may not be defined yet).')
    print('Use: model = build_baseline_lstm(input_shape=(30, n_features))')


c:\Users\LENOVO\miniconda3\envs\dicoding_train\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,513 (17.63 KB)

 Trainable params: 4,513 (17.63 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[es],
    shuffle=False  
)

Epoch 1/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - loss: 0.0317 - mae: 0.1111 - val_loss: 0.0386 - val_mae: 0.1116
Epoch 2/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0175 - mae: 0.0784 - val_loss: 0.0064 - val_mae: 0.0460
Epoch 3/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - loss: 0.0162 - mae: 0.0715 - val_loss: 0.0070 - val_mae: 0.0515
Epoch 4/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0150 - mae: 0.0674 - val_loss: 0.0103 - val_mae: 0.0669
Epoch 5/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0153 - mae: 0.0680 - val_loss: 0.0066 - val_mae: 0.0488
Epoch 6/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0141 - mae: 0.0646 - val_loss: 0.0074 - val_mae: 0.0537
Epoch 7/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0140 - mae: 0.0645 - val_loss: 0.0067 - val_mae: 0.0499
Epoch 8/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - loss: 0.0134 - mae: 0.0629 - val_loss: 0.0245 - val_mae: 0.0990
Epoch 9/100
866/866 ━━━━━━━━━━━━

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# =========================
# Helper RMSE (sklearn lama tidak punya squared=False)
# =========================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Evaluation on test set comparing Naive, 7-day MA, and LSTM baseline
# Requirements: `test_data`, `train_scaled`, `val_scaled`, `test_scaled`, `scaler`, and (optionally) `model` must exist.

# =========================
# 1) Baselines (UNSCALED) on test_data
# =========================
naive_y, naive_pred, ma7_pred = [], [], []

for (region, komoditas), group in test_data.groupby(['region', 'komoditas']):
    g = group.sort_values('tanggal').reset_index(drop=True)
    harga = g['harga'].astype(float).values
    n = len(harga)
    if n < 2:
        continue

    for i in range(n - 1):
        true_next = harga[i + 1]
        naive_y.append(true_next)
        naive_pred.append(harga[i])

        window = harga[max(0, i - 6): i + 1]  # max 7 hari terakhir termasuk hari i
        ma7_pred.append(window.mean())

naive_y = np.asarray(naive_y, dtype=float)
naive_pred = np.asarray(naive_pred, dtype=float)
ma7_pred = np.asarray(ma7_pred, dtype=float)

if len(naive_y) == 0:
    print("No baseline samples could be computed (check test_data grouping / length).")
else:
    naive_mae  = mean_absolute_error(naive_y, naive_pred)
    naive_rmse = rmse(naive_y, naive_pred)

    ma7_mae  = mean_absolute_error(naive_y, ma7_pred)
    ma7_rmse = rmse(naive_y, ma7_pred)

    print('Baseline (Naive) - MAE: {:.4f}, RMSE: {:.4f}'.format(naive_mae, naive_rmse))
    print('Baseline (MA7)   - MAE: {:.4f}, RMSE: {:.4f}\n'.format(ma7_mae, ma7_rmse))

# =========================
# 2) LSTM evaluation (if `model` exists)
# =========================
if 'model' not in globals():
    print('Model not found in workspace. Train the LSTM first to evaluate it on the test set.')
else:
    def make_sequence_with_idx(arr, lookback=30, horizon=1, target_idx=0):
        """
        arr: shape (T, n_features) scaled
        X window: t-lookback ... t-1 (length lookback)
        y: value at t+horizon-1
        idxs: index of y in original series
        """
        arr = np.asarray(arr, dtype=float)
        T = len(arr)

        X, y, idxs = [], [], []
        # t = index "current time" (end-exclusive of window)
        # window = [t-lookback, ..., t-1], y at (t+horizon-1)
        for t in range(lookback, T - horizon + 1):
            X.append(arr[t - lookback: t, :])
            y.append(arr[t + horizon - 1, target_idx])
            idxs.append(t + horizon - 1)

        if len(X) == 0:
            n_features = arr.shape[1] if arr.ndim == 2 else 1
            return np.empty((0, lookback, n_features)), np.empty((0,)), np.empty((0,), dtype=int)

        return np.stack(X), np.asarray(y, dtype=float).reshape(-1), np.asarray(idxs, dtype=int)

    # Gabungkan semua split scaled untuk dapat urutan lengkap per seri
    full_scaled = pd.concat([train_scaled, val_scaled, test_scaled], ignore_index=True)

    lstm_preds = []
    lstm_trues = []

    # Sesuaikan kolom fitur yang scaler-fit (di sini diasumsikan ['harga','ump'])
    FEAT_COLS = ['harga', 'ump']
    n_feat = len(FEAT_COLS)

    for (region, komoditas), group in full_scaled.groupby(['region', 'komoditas']):
        g = group.sort_values('tanggal').reset_index(drop=True)

        # Pastikan kolom ada
        if not all(c in g.columns for c in FEAT_COLS):
            continue

        arr = g[FEAT_COLS].astype(float).values
        dates = g['tanggal'].values

        X_all, y_all, idxs = make_sequence_with_idx(arr, lookback=30, horizon=1, target_idx=0)
        if X_all.shape[0] == 0:
            continue

        # Ambil tanggal test untuk seri ini
        test_dates = test_data[
            (test_data['region'] == region) & (test_data['komoditas'] == komoditas)
        ]['tanggal'].values
        if len(test_dates) == 0:
            continue

        # y index yang jatuh di periode test
        mask = np.isin(dates[idxs], test_dates)
        if not mask.any():
            continue

        X_sel = X_all[mask]
        y_sel = y_all[mask]

        # Predict (scaled)
        y_pred_scaled = model.predict(X_sel, verbose=0).ravel()

        # Inverse transform harga saja (pakai dummy matrix sesuai jumlah fitur)
        tmp = np.zeros((len(y_pred_scaled), n_feat), dtype=float)
        tmp[:, 0] = y_pred_scaled
        pred_inv = scaler.inverse_transform(tmp)[:, 0]

        tmp[:, 0] = y_sel
        true_inv = scaler.inverse_transform(tmp)[:, 0]

        lstm_preds.extend(pred_inv.tolist())
        lstm_trues.extend(true_inv.tolist())

    if len(lstm_preds) == 0:
        print('No LSTM predictions were generated for the test set (maybe lookback crosses series boundaries).')
    else:
        lstm_mae  = mean_absolute_error(lstm_trues, lstm_preds)
        lstm_rmse = rmse(lstm_trues, lstm_preds)

        print('LSTM baseline     - MAE: {:.4f}, RMSE: {:.4f}\n'.format(lstm_mae, lstm_rmse))

        # Compare and warn if LSTM is worse than Naive
        if len(naive_y) > 0 and lstm_mae > naive_mae:
            print('Warning: LSTM MAE > Naive MAE. Common causes: parsing dates, split leakage, scaling leakage, or sequence off-by-one.')

Baseline (Naive) - MAE: 1347.4751, RMSE: 3225.2251
Baseline (MA7)   - MAE: 1664.3606, RMSE: 3770.2668

LSTM baseline     - MAE: 2036.5797, RMSE: 3937.8138



In [23]:
y_pred_scaled = model.predict(X_test, verbose=0).ravel()

# inverse harga saja (StandardScaler aman per-kolom)
tmp = np.zeros((len(y_pred_scaled), len(FEAT_COLS)))
tmp[:, 0] = y_pred_scaled
pred_inv = scaler.inverse_transform(tmp)[:, 0]

tmp[:, 0] = y_test.ravel()
true_inv = scaler.inverse_transform(tmp)[:, 0]

print("pred_inv min/max:", pred_inv.min(), pred_inv.max())
print("true_inv min/max:", true_inv.min(), true_inv.max())

pred_inv min/max: 12338.906855338886 143060.28789050423
true_inv min/max: 11001.0 150000.0


Untuk konteks proyek ini (forecast harga harian per komoditas/region dengan horizon 1 hari), baseline Naive saat ini adalah titik acuan yang paling kuat dan paling layak dijadikan standar minimal, karena pola persistensi “harga besok ≈ harga hari ini” mendominasi data uji; sementara itu, baseline LSTM masih belum memberikan nilai tambah dibanding Naive, sehingga pendekatan yang paling berpeluang menghasilkan peningkatan yang nyata adalah (i) tetap mempertahankan Naive sebagai baseline utama, lalu (ii) menguji model yang secara praktis sering unggul pada time-series pendek dan noisy seperti GRU (lebih stabil/efisien daripada LSTM), SARIMAX (terutama bila memanfaatkan fitur eksogen seperti UMP dan variabel kalender), serta model berbasis fitur lag seperti XGBoost/LightGBM; BiLSTM boleh dicoba untuk pembanding, tetapi untuk forecasting satu arah sering tidak seefektif GRU/SARIMAX dan lebih rentan overfit. Dengan kata lain, jalur paling “worth it” untuk proyek ini adalah mengutamakan GRU dan SARIMAX (plus fitur kalender/lag) sebagai kandidat utama untuk mengalahkan Naive, sambil menjaga pipeline evaluasi yang benar-benar sebanding (aligned) agar perbandingan hasil valid.